# **Homework 2: Phoneme Classification**


Objectives:
* Solve a classification problem with deep neural networks (DNNs).
* Understand recursive neural networks (RNNs).

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

# Some Utility Functions
**Fixes random number generator seeds for reproducibility.**

In [76]:
import numpy as np
import torch
import random

def same_seeds(seed):
    random.seed(seed) 
    np.random.seed(seed)  
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

**Helper functions to pre-process the training data from raw MFCC features of each utterance.**

A phoneme may span several frames and is dependent to past and future frames. \
Hence we concatenate neighboring phonemes for training to achieve higher accuracy. The **concat_feat** function concatenates past and future k frames (total 2k+1 = n frames), and we predict the center frame.

Feel free to modify the data preprocess functions, but **do not drop any frame** (if you modify the functions, remember to check that the number of frames are the same as mentioned in the slides)

In [77]:
import os
import torch
from tqdm import tqdm

def load_feat(path):
    feat = torch.load(path)
    return feat

def shift(x, n):
    if n < 0:
        left = x[0].repeat(-n, 1)
        right = x[:n]
    elif n > 0:
        right = x[-1].repeat(n, 1)
        left = x[n:]
    else:
        return x

    return torch.cat((left, right), dim=0)

def concat_feat(x, concat_n):
    assert concat_n % 2 == 1 # n must be odd
    if concat_n < 2:
        return x
    seq_len, feature_dim = x.size(0), x.size(1)
    x = x.repeat(1, concat_n) 
    x = x.view(seq_len, concat_n, feature_dim).permute(1, 0, 2) # concat_n, seq_len, feature_dim
    mid = (concat_n // 2)
    for r_idx in range(1, mid+1):
        x[mid + r_idx, :] = shift(x[mid + r_idx], r_idx)
        x[mid - r_idx, :] = shift(x[mid - r_idx], -r_idx)

    return x.permute(1, 0, 2).view(seq_len, concat_n * feature_dim)

def preprocess_data(split, feat_dir, phone_path, concat_nframes, train_ratio=0.8):
    class_num = 41 # NOTE: pre-computed, should not need change

    if split == 'train' or split == 'val':
        mode = 'train'
    elif split == 'test':
        mode = 'test'
    else:
        raise ValueError('Invalid \'split\' argument for dataset: PhoneDataset!')

    label_dict = {}
    if mode == 'train':
        for line in open(os.path.join(phone_path, f'{mode}_labels.txt')).readlines():
            line = line.strip('\n').split(' ')
            label_dict[line[0]] = [int(p) for p in line[1:]]
        
        # split training and validation data
        usage_list = open(os.path.join(phone_path, 'train_split.txt')).readlines()
        random.shuffle(usage_list)
        train_len = int(len(usage_list) * train_ratio)
        usage_list = usage_list[:train_len] if split == 'train' else usage_list[train_len:]

    elif mode == 'test':
        usage_list = open(os.path.join(phone_path, 'test_split.txt')).readlines()

    usage_list = [line.strip('\n') for line in usage_list]
    print('[Dataset] - # phone classes: ' + str(class_num) + ', number of utterances for ' + split + ': ' + str(len(usage_list)))

    max_len = 3000000
    X = torch.empty(max_len, 39 * concat_nframes)
    if mode == 'train':
        y = torch.empty(max_len, dtype=torch.long)

    idx = 0
    for i, fname in tqdm(enumerate(usage_list)):
        feat = load_feat(os.path.join(feat_dir, mode, f'{fname}.pt'))
        cur_len = len(feat)
        feat = concat_feat(feat, concat_nframes)
        if mode == 'train':
          label = torch.LongTensor(label_dict[fname])

        X[idx: idx + cur_len, :] = feat
        if mode == 'train':
          y[idx: idx + cur_len] = label

        idx += cur_len

    X = X[:idx, :]
    if mode == 'train':
      y = y[:idx]

    print(f'[INFO] {split} set')
    print(X.shape)
    if mode == 'train':
      print(y.shape)
      return X, y
    else:
      return X


# Dataset

In [78]:
import torch
from torch.utils.data import Dataset

class LibriDataset(Dataset):
    def __init__(self, X, y=None):
        self.data = X
        if y is not None:
            self.label = torch.LongTensor(y)
        else:
            self.label = None

    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]

    def __len__(self):
        return len(self.data)


# Model
Feel free to modify the structure of the model.

In [88]:
import torch.nn as nn

class BasicBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BasicBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        x = self.block(x)
        return x


class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim=41, hidden_layers=1, hidden_dim=256):
        super(Classifier, self).__init__()

        self.fc = nn.Sequential(
            BasicBlock(input_dim, hidden_dim),
            *[BasicBlock(hidden_dim, hidden_dim) for _ in range(hidden_layers)],
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.fc(x)
        return x

# Hyper-parameters

In [89]:
# data prarameters
concat_nframes = 17              # the number of frames to concat with, n must be odd (total 2k+1 = n frames)
train_ratio = 0.75               # the ratio of data used for training, the rest will be used for validation

# training parameters
seed = 1213                        # random seed
batch_size = 512                # batch size
num_epoch = 60                   # the number of training epoch
learning_rate = 1e-4         # learning rate
model_path = './model.ckpt'     # the path where the checkpoint will be saved

# model parameters
input_dim = 39 * concat_nframes # the input dim of the model, you should not change the value
hidden_layers = 8               # the number of hidden layers
hidden_dim = 1024                # the hidden dim

# Dataloader

In [90]:
from torch.utils.data import DataLoader
import gc

same_seeds(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

# preprocess data
train_X, train_y = preprocess_data(split='train', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)
same_seeds(seed) 
val_X, val_y = preprocess_data(split='val', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)

# get dataset
train_set = LibriDataset(train_X, train_y)
val_set = LibriDataset(val_X, val_y)

# remove raw feature to save memory
del train_X, train_y, val_X, val_y
gc.collect()

# get dataloader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

DEVICE: cuda
[Dataset] - # phone classes: 41, number of utterances for train: 2571


2571it [00:06, 401.78it/s]


[INFO] train set
torch.Size([1588590, 663])
torch.Size([1588590])
[Dataset] - # phone classes: 41, number of utterances for val: 858


858it [00:02, 378.27it/s]


[INFO] val set
torch.Size([528204, 663])
torch.Size([528204])


# Training

In [91]:
# create model, define a loss function, and optimizer
model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_acc = 0.0
for epoch in range(num_epoch):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0
    
    # training
    model.train() # set the model to training mode
    for i, batch in enumerate(tqdm(train_loader)):
        features, labels = batch
        features = features.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad() 
        outputs = model(features) 
        
        loss = criterion(outputs, labels)
        loss.backward() 
        optimizer.step() 
        
        _, train_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        train_acc += (train_pred.detach() == labels.detach()).sum().item()
        train_loss += loss.item()
    
    # validation
    model.eval() # set the model to evaluation mode
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader)):
            features, labels = batch
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            
            loss = criterion(outputs, labels) 
            
            _, val_pred = torch.max(outputs, 1) 
            val_acc += (val_pred.cpu() == labels.cpu()).sum().item() # get the index of the class with the highest probability
            val_loss += loss.item()

    print(f'[{epoch+1:03d}/{num_epoch:03d}] Train Acc: {train_acc/len(train_set):3.5f} Loss: {train_loss/len(train_loader):3.5f} | Val Acc: {val_acc/len(val_set):3.5f} loss: {val_loss/len(val_loader):3.5f}')

    # if the model improves, save a checkpoint at this epoch
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f'saving model with acc {best_acc/len(val_set):.5f}')


100%|██████████| 1032/1032 [00:03<00:00, 260.73it/s]


[001/060] Train Acc: 0.49516 Loss: 1.71348 | Val Acc: 0.60494 loss: 1.28526
saving model with acc 0.60494


100%|██████████| 1032/1032 [00:03<00:00, 259.19it/s]


[002/060] Train Acc: 0.58496 Loss: 1.35763 | Val Acc: 0.64129 loss: 1.15743
saving model with acc 0.64129


100%|██████████| 1032/1032 [00:04<00:00, 243.77it/s]


...

[058/060] Train Acc: 0.75661 Loss: 0.75137 | Val Acc: 0.74339 loss: 0.81785
saving model with acc 0.74339


100%|██████████| 1032/1032 [00:03<00:00, 264.49it/s]


[059/060] Train Acc: 0.75734 Loss: 0.74876 | Val Acc: 0.74351 loss: 0.81634
saving model with acc 0.74351


100%|██████████| 1032/1032 [00:03<00:00, 259.32it/s]

[060/060] Train Acc: 0.75834 Loss: 0.74681 | Val Acc: 0.74325 loss: 0.81905


In [92]:
del train_set, val_set
del train_loader, val_loader
gc.collect()

0

# Testing
Create a testing dataset, and load model from the saved checkpoint.

In [84]:
# load data
test_X = preprocess_data(split='test', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes)
test_set = LibriDataset(test_X, None)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

[Dataset] - # phone classes: 41, number of utterances for test: 857


857it [00:02, 384.83it/s]

[INFO] test set
torch.Size([527364, 663])


In [85]:
# load model
model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

Make prediction.

In [86]:
pred = np.array([], dtype=np.int32)

model.eval()
with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader)):
        features = batch
        features = features.to(device)

        outputs = model(features)

        _, test_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        pred = np.concatenate((pred, test_pred.cpu().numpy()), axis=0)


100%|██████████| 1031/1031 [00:03<00:00, 284.06it/s]


Write prediction to a CSV file.

After finish running this block, download the file `prediction.csv` from the files section on the left-hand side and submit it to Kaggle.

In [87]:
with open('prediction.csv', 'w') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(pred):
        f.write('{},{}\n'.format(i, y))

# 我的实验记录

每次改了超参数就在下面手动追加一行，记录改了什么、Val Acc 多少、我的猜测对不对。

In [ ]:
# 格式：改了什么 -> Val Acc -> 猜测是否成立
# 示例：concat_nframes 3->3 (基线) -> Val Acc 0.50041

log = []

log.append({
    "改动": "基线 concat_nframes=3, hidden_layers=2, hidden_dim=64, epoch=10",
    "val_acc": 0.50041,
    "猜测": "",
    "结果是否符合猜测": "",

    "第一次改动": "concat_nframes3->5",
    "val_acc": 0.53863,
    "猜测": "better",
    "结果是否符合猜测": "yes",

    "第二次改动": "hidden_layer2->4",
    "val_acc": 0.53188,
    "猜测": "better",
    "结果是否符合猜测": "并没有",

    "第三次改动": "hidden_layer2->4, hidden_dim=128",
    "val_acc": 0.56636,
    "猜测": "better",
    "结果是否符合猜测": "yes",

    "第四次改动": "hidden_layer2->4, hidden_dim=128, num_epoch10->40, learning rate 1e-4->1e-5",
    "val_acc": 0.53447,
    "猜测": "better",
    "结果是否符合猜测": "not yet  怀疑是因为学习率太低轮次太少不能到达最低点，同时时间花费太多",

    "第五次改动": "hidden_dim=256->512, concat_nframes = 9->15",
    "val_acc": 0.80434,
    "猜测": "better",
    "结果是否符合猜测": "好的批爆！",

    "第六次改动": "hidden_layer4->6",
    "val_acc": 0.81143,
    "猜测": "better",
    "结果是否符合猜测": "变好效果一般",

    "第七次改动": "learning_rate 1e-4->1e-5, epoch->60",
    "val_acc": 0.69414,
    "猜测": "better",
    "结果是否符合猜测": "反而变得更差，调高epoch",

    "第八次改动": "epoch->100",
    "val_acc": 0.72024,
    "猜测": "better",
    "结果是否符合猜测": "也很一般，时间太久半个小时",

    "第九次改动": "epoch->60, learning_rate=1e-4, hidden_layer->8, hidden_dim->1024",
    "val_acc": 0.90734,
    "猜测": "better",
    "结果是否符合猜测": "非常不错！",

    #模型出现了过拟合现象，而且争取率和kaggle不一样，先进行改动使得正确率是真的

    "第十次改动": "model一格加上BatchNormld",
    "val_acc": 0.68337,
    "猜测": "better",
    "结果是否符合猜测": "也很一般，时间太久半个小时，BN收益不明显",

    "第十一次改动": "model一格再加上dropout",
    "val_acc": 0.74325,
    "猜测": "better",
    "结果是否符合猜测": "趋势明显，时间半个小时，增加近六个点",

    "第十二次改动": "从MLP改成RNN_____新建副本——hw2-RNN",
})


for r in log:
    print(r)


{'改动': '基线 concat_nframes=3, hidden_layers=2, hidden_dim=64, epoch=10', 'val_acc': 0.72024, '猜测': 'better', '结果是否符合猜测': '也很一般', '第一次改动': 'concat_nframes3->5', '第二次改动': 'hidden_layer2->4', '第三次改动': 'hidden_layer2->4, hidden_dim=128', '第四次改动': 'hidden_layer2->4, hidden_dim=128, num_epoch10->40, learning rate 1e-4->1e-5', '第五次改动': 'hidden_dim=256->512, concat_nframes = 9->15', '第六次改动': 'hidden_layer4->6', '第七次改动': 'epoch->100'}
